# Phase 1 — ICC + partial-pooling shrinkage: is durable player level enough?

**After the baselines: how much of weekly points is durable player _level_, and can shrinking toward it out-rank the raw mean?**

_Read-only checkpoint — Phase 1 of [docs/predictive-layer-plan.md](../../docs/predictive-layer-plan.md). Two deliverables: **ICC inference** measures the between/within split as a model parameter (inferential, contemporaneous — no lag); **EB shrinkage** tests whether empirical-Bayes shrinkage beats the raw expanding mean on the Phase-0 walk-forward harness (predictive)._
Population: `minutes > 0`, **DGW excluded**, per position, players with **≥ 10 games** (ICC inference, whole season) / evaluate GW > 3 walk-forward (EB shrinkage).

> **How to read.** The ICC fit is a random-intercept model per position that reports **ICC** — the share of weekly-points variance that is stable between-player level — with a bootstrap CI and a likelihood-ratio test, reconciled against Q1's descriptive sum-of-squares share, and stress-tested for normality with a distribution-free between-share bootstrap. EB shrinkage builds a leakage-safe shrunk estimate (`lvl_shrunk = μ_pos + λ·(mean_i − μ_pos)`) and scores it within position against the raw mean. All numbers come from `research.kernels` and `model.eval` — the notebook only renders them.

## Setup
> Load the mart, apply the population filter, and compute the deliverables: ICC per position + Q1 SS between-share for reconciliation + the distribution-free normality-robustness bootstrap; and EB shrinkage (shrink vs raw mean, per position).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.variance_components import decompose_variance
from research.kernels.inferential.variance_components import mixed_effects_icc, between_share_bootstrap
from model.forecast.shrinkage import score_shrinkage_by_position
from model.eval.walkforward import POSITIONS

try:
    loaded = load_mart()
except (MartNotBuiltError, MartSchemaError) as err:
    print(f'Rebuilding mart ({type(err).__name__})...'); run_pipeline(force=True); loaded = load_mart()
mart = loaded.mart
pop = mart[(mart['minutes'] > 0) & (~mart['is_dgw'].astype(bool))]

# ICC per position (whole season) + Q1 descriptive SS between-share + distribution-free
# normality-robustness bootstrap (a Gaussian-free CI on the between-share).
icc_rows = []
for pos in POSITIONS:
    r = mixed_effects_icc(pop[pop['position'] == pos], n_bootstrap=300)
    ss = decompose_variance(pop[pop['position'] == pos])
    boot_lo, boot_med, boot_hi = between_share_bootstrap(pop[pop['position'] == pos])
    icc_rows.append({
        'position': pos, 'icc': r['icc'], 'icc_lo': r['icc_ci_lo'], 'icc_hi': r['icc_ci_hi'],
        'ss_between': ss['pct_between'] / 100.0,
        'boot_lo': boot_lo, 'boot_med': boot_med, 'boot_hi': boot_hi,
        'sigma2_between': r['sigma2_between'], 'sigma2_within': r['sigma2_within'],
        'lrt_p': r['lrt_p'], 'n_players': r['n_players'],
    })
icc_tbl = pd.DataFrame(icc_rows).set_index('position').reindex(POSITIONS)

# EB shrinkage — shrink vs raw mean, per position, walk-forward.
shrink_tbl = score_shrinkage_by_position(mart)

## (a) ICC — how much of weekly points is durable player level?
> ICC is small everywhere (**GK ≈ 0**, outfield ~0.06–0.10) but the LRT rejects the pooled null decisively for DEF/MID/FWD — the between-player level is *real but thin*. ICC sits just **below** Q1's SS between-share (red diamonds) at every position: the expected unbalanced-panel gap (the SS-share is inflated by finite-group-mean dispersion; the variance component corrects it). Same ordering — reconciliation passes. **GK is genuine noise** (ICC 0, LRT p = 0.5).
>
> **Normality robustness.** The `boot_*` columns are a distribution-free player-clustered bootstrap of the SS between-share — a normality-free CI that confirms the ICC's ordering/magnitude without the Gaussian assumption. GK's SS-share 0.038 [0.018, 0.063] excludes 0 by small-sample bias; the variance-component ICC correctly gives 0 (the LMM is the trustworthy read there).

In [ ]:
display(icc_tbl[['icc', 'icc_lo', 'icc_hi', 'ss_between', 'boot_lo', 'boot_med', 'boot_hi', 'sigma2_between', 'sigma2_within', 'lrt_p', 'n_players']].round(3))

y = np.arange(len(POSITIONS))
icc = icc_tbl['icc']
lo = (icc - icc_tbl['icc_lo']).clip(lower=0)
hi = (icc_tbl['icc_hi'] - icc).clip(lower=0)
fig, ax = plt.subplots(figsize=(8.5, 3.4))
ax.barh(y, icc.values, color='#1f77b4', alpha=0.85, label='ICC (model)')
ax.errorbar(icc.values, y, xerr=[lo.values, hi.values], fmt='none', ecolor='#08306b', capsize=4, lw=1.2)
ax.scatter(icc_tbl['ss_between'].values, y, color='#d62728', zorder=5, marker='D', s=45, label='Q1 SS between-share')
for i, pos in enumerate(POSITIONS):
    ax.text(max(icc_tbl['icc_hi'][pos], icc_tbl['ss_between'][pos]) + 0.006, i, f"LRT p={icc_tbl['lrt_p'][pos]:.0e}", fontsize=7, color='#333', va='center')
ax.set_yticks(y); ax.set_yticklabels(POSITIONS)
ax.set_xlabel('share of weekly-points variance that is between-player (durable level)')
ax.set_title('ICC reconciles with Q1 SS between-share (ICC ≤ SS: unbalanced-panel gap)')
ax.legend(loc='lower right', fontsize=8); ax.set_xlim(0, float(np.nanmax(icc_tbl['icc_hi'])) + 0.06)
plt.tight_layout(); plt.show()

## (b) EB shrinkage — does shrinking toward the mean out-rank the raw mean?
> **No.** EB shrinkage is *slightly worse* than the raw expanding mean on within-position Spearman at every position (precision@k / NDCG are within noise). Because the between-player slice is thin and λ varies by games-played, shrinkage **reorders players by sample size** — adding noise to the rank, not signal. Rank correlation is insensitive to the variance reduction shrinkage buys, so there is little upside and a small cost.
>
> The 95% **block-bootstrap CIs** (added in the Phase-1 cleanup) overlap between mean and shrunk at every position — the null is *shown*, not merely asserted. GK is degenerate: ICC(GK)=0 → λ→0 → `lvl_shrunk` collapses to the position mean, so its row is not a genuine model comparison (and GK's true naive bar is rolling-5, not `base_season`).

In [ ]:
display(shrink_tbl[['spearman', 'ci_lo', 'ci_hi', 'precision_at_k', 'ndcg_at_k', 'coverage', 'n_gw']].round(4))

mean_lbl, shr_lbl = 'mean (incumbent)', 'EB shrunk (Phase 1)'
sp = shrink_tbl['spearman'].unstack('estimator').reindex(POSITIONS)
lo = shrink_tbl['ci_lo'].unstack('estimator').reindex(POSITIONS)
hi = shrink_tbl['ci_hi'].unstack('estimator').reindex(POSITIONS)
def _err(lbl):  # asymmetric yerr = [point - ci_lo, ci_hi - point], clipped at 0
    return [(sp[lbl] - lo[lbl]).clip(lower=0).values, (hi[lbl] - sp[lbl]).clip(lower=0).values]
x = np.arange(len(POSITIONS)); w = 0.36
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(x - w / 2, sp[mean_lbl].values, w, color='#1f77b4', label='raw mean (incumbent)',
       yerr=_err(mean_lbl), capsize=3, ecolor='#08306b', error_kw={'lw': 1.0})
ax.bar(x + w / 2, sp[shr_lbl].values, w, color='#ff7f0e', label='EB shrunk (Phase 1)',
       yerr=_err(shr_lbl), capsize=3, ecolor='#7f3f00', error_kw={'lw': 1.0})
for i in range(len(POSITIONS)):
    ax.text(x[i] - w / 2, hi[mean_lbl].values[i] + 0.006, f"{sp[mean_lbl].values[i]:.3f}", ha='center', fontsize=7)
    ax.text(x[i] + w / 2, hi[shr_lbl].values[i] + 0.006, f"{sp[shr_lbl].values[i]:.3f}", ha='center', fontsize=7)
ax.axhline(0, color='#999', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(POSITIONS)
ax.set_ylabel('within-position Spearman (higher = better)')
ax.set_title('EB shrinkage does NOT out-rank the raw mean; 95% block-bootstrap CIs overlap everywhere')
ax.legend(loc='upper left', fontsize=8)
ax.set_ylim(min(0.0, float(np.nanmin(lo.values)) - 0.02), float(np.nanmax(hi.values)) + 0.06)
plt.tight_layout(); plt.show()

## Summary — Phase 1 verdict

| gate | criterion | result |
|---|---|---|
| 1 | ICC reconciles with Q1 SS between-share (magnitude + order) | ✅ pass |
| 2 | σ²_between real where Q1 found it (LRT p < 0.05) | ✅ pass (GK correctly null) |
| 3 | `lvl_shrunk` out-ranks raw mean per position | ❌ fail — slightly worse everywhere |

**Outcome (pre-registered fallback): ship the ICC inference, record EB shrinkage as an honest null.** The ICC inference — the inferential formalization of Q1 — is the durable Phase-1 output: durable player level is a small (~6–10% outfield, ~0 GK) but statistically real share of weekly-points variance, robust to dropping normality. EB shrinkage confirms that a level-only shrinkage ranker cannot beat the raw mean on a single season; kept in the codebase for a cross-season re-run (Phase 6). Frozen numbers: [docs/studies/results/predictive-phase1-icc-shrinkage.md](../../docs/studies/results/predictive-phase1-icc-shrinkage.md).

**Where next.** Durable identity is a thin slice with a low ceiling — Phase 2 adds features (opponent, minutes, fixtures, xG) to reach into the large within-player slice that identity alone cannot explain.